# 00 — Verify Connection

Confirms Databricks Connect is working against Serverless compute and that the
Unity Catalog target schemas exist (or creates them). This demo uses three schemas:
- `{UC_SCHEMA}_bronze` — raw ingested data
- `{UC_SCHEMA}_silver` — cleaned, deduplicated tables
- `{UC_SCHEMA}_gold` — star schema for analytics

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()
print("Spark version:", spark.version)

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Spark version: 4.1.0


In [2]:
spark.sql("SELECT current_user(), current_timestamp()").show(truncate=False)

+------------------------------+-------------------------+
|current_user()                |current_timestamp()      |
+------------------------------+-------------------------+
|alexander.booth@databricks.com|2026-05-28 16:48:21.64425|
+------------------------------+-------------------------+



In [3]:
UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "nil_sponsorships")

BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}")
    print(f"Schema ready: {UC_CATALOG}.{schema}")

print(f"\nTarget catalog: {UC_CATALOG}")
spark.sql(f"DESCRIBE SCHEMA {UC_CATALOG}.{BRONZE_SCHEMA}").show(truncate=False)

Schema ready: alexander_booth.nil_sponsorships_bronze


Schema ready: alexander_booth.nil_sponsorships_silver


Schema ready: alexander_booth.nil_sponsorships_gold

Target catalog: alexander_booth


+-------------------------+------------------------------+
|database_description_item|database_description_value    |
+-------------------------+------------------------------+
|Catalog Name             |alexander_booth               |
|Namespace Name           |nil_sponsorships_bronze       |
|Comment                  |                              |
|Location                 |                              |
|Owner                    |alexander.booth@databricks.com|
+-------------------------+------------------------------+



## Optional: Reset

Uncomment and run the cell below to drop all schemas and start fresh.

In [4]:
# # Uncomment to reset everything
# for schema in [GOLD_SCHEMA, SILVER_SCHEMA, BRONZE_SCHEMA]:
#     spark.sql(f"DROP SCHEMA IF EXISTS {UC_CATALOG}.{schema} CASCADE")
#     print(f"Dropped: {UC_CATALOG}.{schema}")

# # Recreate empty schemas
# for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
#     spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}")
#     print(f"Recreated: {UC_CATALOG}.{schema}")